



## Cellule 1 — Imports & Ingestion des données

### Objectif
Charger les 5 fichiers CSV de façon professionnelle : gestion de l'encodage, optimisation des `dtypes`, et affichage des **shapes** à chaque étape pour tracer les éventuelles pertes.

---

### Stratégie d'Architecture & Jointures

Notre pipeline repose sur **trois couches fonctionnelles** :

| Couche | Fichier(s) | Rôle | Clé de jointure |
|---|---|---|---|
| **Interactions (CF)** | `collaborative_books_df.csv` | Table centrale — notes réelles + mappings entiers | `book_id_mapping`, `user_id_mapping` |
| **Métadonnées (CB)** | `collaborative_book_metadata.csv` | Texte riche pour TF-IDF (desc, genre, auteur) | `book_id_mapping` → JOIN LEFT sur la couche CF |
| **Décodage identités** | `user_id_map.csv`, `book_id_map.csv`, `book_titles.csv` | Rétablir UUIDs / titres en sortie finale | `user_id_csv`, `book_id_csv` → `book_id` |

**Pourquoi cette stratégie ?**
- Les colonnes `*_mapping` sont déjà des **entiers continus 0-based** → zero-cost pour construire les matrices `scipy.sparse` sans re-encodage.
- On ne joint **jamais** les tables de décodage dans la table de travail : elles sont gardées séparées et appelées uniquement à l'affichage.
- `book_titles.csv` (1.4M lignes) est converti directement en `dict` puis supprimé → aucun DataFrame inutile en mémoire.

---

### Lecture rapide
- présence des fichiers
- colonnes clés bien chargées
- taux de recouvrement entre interactions et métadonnées

In [ ]:
# =============================================================================
# CELLULE 1 — Imports + Chargement simple des fichiers CSV
# =============================================================================

# 1) Imports
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import sparse
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

print(" Librairies importées")
print(f"   NumPy: {np.__version__} | Pandas: {pd.__version__}")

# 2) Chemins des fichiers
DATA_DIR = Path("Dataset")
PATHS = {
    "interactions": DATA_DIR / "collaborative_books_df.csv",
    "metadata": DATA_DIR / "collaborative_book_metadata.csv",
    "book_id_map": DATA_DIR / "book_id_map.csv",
    "user_id_map": DATA_DIR / "user_id_map.csv",
    "book_titles": DATA_DIR / "book_titles.csv",
}

# Vérifier que tous les fichiers existent
missing_files = [name for name, path in PATHS.items() if not path.exists()]
if missing_files:
    print(f" Fichiers manquants: {missing_files}")
else:
    print(" Tous les fichiers sont présents\n")

# 3) Charger la table principale: interactions
# Cette table contient les notes réelles utilisateur-livre.
interaction_cols = ["user_id_mapping", "book_id_mapping", "book_id", "title", "Actual Rating"]
interaction_dtypes = {
    "user_id_mapping": "int32",
    "book_id_mapping": "int32",
    "book_id": "int32",
    "title": "category",
    "Actual Rating": "int8",
}

print("── Chargement: collaborative_books_df.csv")
interactions_raw = pd.read_csv(
    PATHS["interactions"],
    usecols=interaction_cols,
    dtype=interaction_dtypes,
    encoding="utf-8",
)

loaded_interaction_cols = set(interactions_raw.columns)
expected_interaction_cols = set(interaction_cols)
missing_interaction_cols = sorted(expected_interaction_cols - loaded_interaction_cols)
if missing_interaction_cols:
    print(f" Colonnes manquantes dans interactions: {missing_interaction_cols}")
else:
    print(" Colonnes interactions OK")

print(f"Shape interactions: {interactions_raw.shape}")
print(
    f"Plage user_id_mapping: [{interactions_raw['user_id_mapping'].min()} - "
    f"{interactions_raw['user_id_mapping'].max()}]"
)
print(
    f"Plage book_id_mapping: [{interactions_raw['book_id_mapping'].min()} - "
    f"{interactions_raw['book_id_mapping'].max()}]"
)
print(
    f"Plage Actual Rating: [{interactions_raw['Actual Rating'].min()} - "
    f"{interactions_raw['Actual Rating'].max()}]"
)
print()

# 4) Charger les métadonnées livres (utile pour le Content-Based)
metadata_cols = [
    "book_id_mapping", "book_id", "title", "description", "genre",
    "name", "num_pages", "ratings_count", "image_url", "url"
]
metadata_dtypes = {
    "book_id_mapping": "int32",
    "book_id": "int32",
    "num_pages": "float32",
    "ratings_count": "int32",
}

print("── Chargement: collaborative_book_metadata.csv")
metadata_raw = pd.read_csv(
    PATHS["metadata"],
    usecols=metadata_cols,
    dtype=metadata_dtypes,
    encoding="utf-8",
)

if "book_id_mapping" not in metadata_raw.columns:
    print(" Clé 'book_id_mapping' absente des métadonnées")
else:
    print(" Clé de jointure metadata OK")
print(f"Shape metadata: {metadata_raw.shape}\n")

# 5) Charger les tables de mapping (utiles pour décodage / affichage final)
print("── Chargement: user_id_map.csv")
user_id_map = pd.read_csv(
    PATHS["user_id_map"],
    dtype={"user_id_csv": "int32", "user_id": "object"},
    encoding="utf-8",
)
print(f"Shape user_id_map: {user_id_map.shape}")

print("── Chargement: book_id_map.csv")
book_id_map = pd.read_csv(
    PATHS["book_id_map"],
    dtype={"book_id_csv": "int32", "book_id": "int32"},
    encoding="utf-8",
)
print(f"Shape book_id_map: {book_id_map.shape}")

print("── Chargement: book_titles.csv -> dict")
titles_tmp = pd.read_csv(
    PATHS["book_titles"],
    usecols=["title", "book_id"],
    dtype={"book_id": "int32"},
    encoding="utf-8",
)
BOOK_TITLE_LOOKUP = dict(zip(titles_tmp["book_id"], titles_tmp["title"]))
del titles_tmp
print(f"Entrées dans BOOK_TITLE_LOOKUP: {len(BOOK_TITLE_LOOKUP):,}\n")

# 6) Vérifier le recouvrement interactions <-> metadata
# Question: combien de livres de la table interactions ont aussi des métadonnées ?
interaction_books = set(interactions_raw["book_id_mapping"].unique())
metadata_books = set(metadata_raw["book_id_mapping"].unique())
overlap_keys = interaction_books & metadata_books

n_books_interactions = len(interaction_books)
n_books_metadata = len(metadata_books)
coverage_pct = (len(overlap_keys) / n_books_interactions) * 100 if n_books_interactions else 0.0

print("── Recouvrement interactions ↔ metadata")
print(f"Livres uniques interactions: {n_books_interactions}")
print(f"Livres uniques metadata:     {n_books_metadata}")
print(f"Intersection:                {len(overlap_keys)} ({coverage_pct:.1f}% couverts)")
print()

# 7) Petit résumé final
print("=" * 55)
print("RÉCAPITULATIF — ÉTAT INITIAL DES DONNÉES")
print("=" * 55)
summary_data = {
    "Table": ["interactions", "metadata", "user_id_map", "book_id_map"],
    "Lignes": [len(interactions_raw), len(metadata_raw), len(user_id_map), len(book_id_map)],
    "Colonnes": [
        interactions_raw.shape[1],
        metadata_raw.shape[1],
        user_id_map.shape[1],
        book_id_map.shape[1],
    ],
}
print(pd.DataFrame(summary_data).to_string(index=False))
print("=" * 55)

---

## Cellule 2 — Fusion & Architecture des tables de travail

### Objectif
Construire les **deux DataFrames de travail** qui alimenteront l'ensemble du pipeline :

| DataFrame | Contenu | Usage |
|---|---|---|
| `df_interactions` | Interactions nettoyées (user × book × note) | Filtrage de sparsité, matrice CF, MF/SVD |
| `df_content` | Métadonnées enrichies (une ligne par livre) | Corpus TF-IDF, Content-Based |

### Stratégie de jointure
- **LEFT JOIN** `interactions_raw` ← `metadata_raw` sur `book_id_mapping`  
  → On conserve **toutes** les interactions, même si un livre n'a pas de métadonnées (NULL pour le CB uniquement).  
  → On trace la perte : lignes avant vs après, nombre de livres sans métadonnées.
- `df_content` = `metadata_raw` dédupliqué sur `book_id_mapping` (une ligne = un livre unique).

### Vérification rapide
- comparaison simple des lignes avant/après JOIN
- comptage des livres avec et sans métadonnées
- aperçu final de `df_interactions` et `df_content`

In [ ]:
# =============================================================================
# CELLULE 2 — Fusion simple des tables (version facile)
# =============================================================================

# 1) Enlever les doublons metadata (1 livre = 1 ligne)
# Si un livre apparait plusieurs fois, on garde la première ligne.
meta_before = len(metadata_raw)
metadata_dedup = metadata_raw.drop_duplicates(subset="book_id_mapping", keep="first")
meta_after = len(metadata_dedup)

print("── Étape 1: Déduplication metadata")
print(f"Avant: {meta_before:,} | Après: {meta_after:,} | Doublons retirés: {meta_before - meta_after:,}")
print()

# 2) Fusion LEFT JOIN : on garde TOUTES les interactions
# Même si un livre n'a pas de metadata, la ligne interaction reste présente.
meta_cols_for_join = ["book_id_mapping", "description", "genre", "name", "num_pages", "ratings_count", "image_url", "url"]
rows_before_join = len(interactions_raw)

df_interactions = interactions_raw.merge(
    metadata_dedup[meta_cols_for_join],
    on="book_id_mapping",
    how="left",
)
rows_after_join = len(df_interactions)

print("── Étape 2: LEFT JOIN interactions + metadata")
print(f"Lignes avant JOIN: {rows_before_join:,}")
print(f"Lignes après JOIN: {rows_after_join:,}")
if rows_after_join == rows_before_join:
    print(" Verification rapide: le LEFT JOIN a gardé le même nombre de lignes")
else:
    print(" Verification rapide: le nombre de lignes a changé après le JOIN")
print()

# 3) Mesurer la couverture metadata dans df_interactions
# Un livre sans metadata a souvent description = NaN.
books_total = df_interactions["book_id_mapping"].nunique()
books_no_meta = df_interactions[df_interactions["description"].isna()]["book_id_mapping"].nunique()
books_with_meta = books_total - books_no_meta
coverage = (books_with_meta / books_total) * 100 if books_total else 0.0

print("── Étape 3: Couverture metadata")
print(f"Livres total: {books_total:,}")
print(f"Livres avec metadata: {books_with_meta:,} ({coverage:.1f}%)")
print(f"Livres sans metadata: {books_no_meta:,} ({100 - coverage:.1f}%)")
print()

# 4) Renommer les colonnes pour la suite du notebook
# Plus simple: rating + author

df_interactions = df_interactions.rename(columns={
    "Actual Rating": "rating",
    "name": "author",
})

print("── Étape 4: Colonnes finales df_interactions")
print(df_interactions.columns.tolist())
print(f"Shape df_interactions: {df_interactions.shape}")
print()

# 5) Construire df_content (1 ligne par livre)
# Table dédiée au Content-Based: texte + infos livre.
df_content = metadata_dedup.copy()

# Si le titre metadata est vide, on récupère le titre depuis interactions
title_from_interactions = (
    interactions_raw[["book_id_mapping", "title"]]
    .drop_duplicates(subset="book_id_mapping")
    .set_index("book_id_mapping")["title"]
    .astype(str)
)

df_content["title"] = df_content["title"].fillna(df_content["book_id_mapping"].map(title_from_interactions))
df_content = df_content.rename(columns={"name": "author"})

print("── Étape 5: df_content prêt")
print(f"Shape df_content: {df_content.shape}")
print(df_content.columns.tolist())
print()

# 6) Résumé final de la cellule
print("=" * 60)
print("RÉSUMÉ CELLULE 2")
print("=" * 60)
summary = {
    "Table": ["df_interactions", "df_content"],
    "Lignes": [len(df_interactions), len(df_content)],
    "Colonnes": [df_interactions.shape[1], df_content.shape[1]],
    "Usage": ["CF / MF / évaluation", "Content-Based (TF-IDF)"],
}
print(pd.DataFrame(summary).to_string(index=False))
print("=" * 60)

---

## Cellule 3 — Nettoyage des données 

### Objectif
Nettoyer les 2 tables de travail pour éviter les erreurs dans les modèles.

### Table 1 : `df_interactions`
On corrige 3 problèmes :
- NaN dans les colonnes clés (`user_id_mapping`, `book_id_mapping`, `rating`)
- notes hors plage [1, 5]
- doublons `(user_id_mapping, book_id_mapping)`

### Table 2 : `df_content`
On corrige les colonnes texte et numériques :
- `title` manquant -> suppression de la ligne
- `author` manquant -> "unknown"
- `description` -> nettoyage texte (HTML, espaces)
- `genre` -> normalisation texte
- `num_pages` invalide -> remplacement par la médiane

### Vérification rapide
- résumé des lignes supprimées
- compte simple des NaN restants
- shape finale des 2 tables

In [ ]:
# =============================================================================
# CELLULE 3 — Nettoyage des données (version simple)
# =============================================================================
import re

# ----------------------------------------------------------------------------
# A) Nettoyage de df_interactions
# ----------------------------------------------------------------------------
print("=" * 60)
print("A) Nettoyage de df_interactions")
print("=" * 60)

shape_0 = df_interactions.shape
print(f"Shape initiale: {shape_0}")

# A1) Supprimer les lignes avec NaN dans les colonnes clés
key_cols = ["user_id_mapping", "book_id_mapping", "rating"]
nan_key_mask = df_interactions[key_cols].isna().any(axis=1)
n_nan_keys = int(nan_key_mask.sum())
df_interactions = df_interactions.loc[~nan_key_mask].copy()
print(f"A1 - Lignes supprimées (NaN colonnes clés): {n_nan_keys}")
print(f"     Shape: {df_interactions.shape}")

# A2) Supprimer les notes hors [1, 5]
invalid_rating_mask = ~df_interactions["rating"].between(1, 5)
n_invalid = int(invalid_rating_mask.sum())
df_interactions = df_interactions.loc[~invalid_rating_mask].copy()
print(f"A2 - Lignes supprimées (notes invalides): {n_invalid}")
print(f"     Shape: {df_interactions.shape}")

# A3) Supprimer les doublons (user, book) en gardant la dernière note
n_before_dedup = len(df_interactions)
df_interactions = df_interactions.drop_duplicates(
    subset=["user_id_mapping", "book_id_mapping"],
    keep="last",
)
n_removed_dup = n_before_dedup - len(df_interactions)
print(f"A3 - Doublons supprimés (user, book): {n_removed_dup}")
print(f"     Shape: {df_interactions.shape}")

# A4) Verification rapide final interactions
remaining_nan_keys = int(df_interactions[key_cols].isna().sum().sum())
remaining_invalid = int((~df_interactions["rating"].between(1, 5)).sum())
remaining_dups = int(df_interactions.duplicated(subset=["user_id_mapping", "book_id_mapping"]).sum())
print(
    "A4 - Verification rapide interactions: "
    f"NaN clés={remaining_nan_keys}, "
    f"ratings hors plage={remaining_invalid}, "
    f"doublons={remaining_dups}"
)

print(" df_interactions nettoyé")
print(f"   Lignes supprimées au total: {shape_0[0] - len(df_interactions)}")

# ----------------------------------------------------------------------------
# B) Nettoyage de df_content
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("B) Nettoyage de df_content")
print("=" * 60)

content_cols = ["title", "author", "description", "genre", "num_pages"]
print("NaN avant nettoyage:")
print(df_content[content_cols].isna().sum().to_string())

shape_b0 = df_content.shape
print(f"\nShape initiale: {shape_b0}")

# B1) Supprimer livres sans titre
n_no_title = int(df_content["title"].isna().sum())
df_content = df_content.dropna(subset=["title"]).copy()
print(f"B1 - Lignes supprimées (title manquant): {n_no_title}")
print(f"     Shape: {df_content.shape}")

# B2) Auteur manquant -> 'unknown'
df_content["author"] = df_content["author"].fillna("unknown").astype(str).str.strip()
print("B2 - author nettoyé (NaN -> 'unknown')")

# B3) Nettoyage description
# On garde les lignes même si description vide (utile pour coverage global).
def clean_text(text) -> str:
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)   # retire balises HTML
    text = re.sub(r"&\w+;", " ", text)     # retire entités HTML
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_content["description"] = df_content["description"].apply(clean_text)
n_empty_desc = int((df_content["description"] == "").sum())
print(f"B3 - descriptions vides après nettoyage: {n_empty_desc}")

# B4) Nettoyage genre
def clean_genre(genre) -> str:
    if pd.isna(genre) or str(genre).strip() in ("", "[]", "nan"):
        return ""
    g = str(genre)
    g = re.sub(r"[\[\]\"'`]", " ", g)
    g = re.sub(r"[,;]", " ", g)
    g = re.sub(r"\s+", " ", g).strip().lower()
    return g

df_content["genre"] = df_content["genre"].apply(clean_genre)
n_empty_genre = int((df_content["genre"] == "").sum())
print(f"B4 - genres vides après nettoyage: {n_empty_genre}")

# B5) Nettoyage num_pages
valid_pages = df_content.loc[df_content["num_pages"] > 0, "num_pages"]
median_pages = valid_pages.median()
n_bad_pages = int(((df_content["num_pages"].isna()) | (df_content["num_pages"] <= 0)).sum())
df_content["num_pages"] = (
    df_content["num_pages"]
    .where(df_content["num_pages"] > 0, other=np.nan)
    .fillna(median_pages)
)
print(f"B5 - num_pages corrigé: {n_bad_pages} valeurs remplacées par la médiane ({median_pages:.0f})")

# B6) Verification rapide final content
remaining_title_nan = int(df_content["title"].isna().sum())
remaining_author_nan = int(df_content["author"].isna().sum())
remaining_pages_nan = int(df_content["num_pages"].isna().sum())
print(
    "B6 - Verification rapide content: "
    f"title NaN={remaining_title_nan}, "
    f"author NaN={remaining_author_nan}, "
    f"num_pages NaN={remaining_pages_nan}"
)

print("\nNaN après nettoyage:")
nan_after = df_content[content_cols].isna().sum()
print(nan_after.to_string())

print(f"\n df_content nettoyé - shape finale: {df_content.shape}")
print(f"   Lignes supprimées au total: {shape_b0[0] - len(df_content)}")

# ----------------------------------------------------------------------------
# Résumé final
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("RÉCAPITULATIF — APRÈS NETTOYAGE")
print("=" * 60)
recap_clean = {
    "DataFrame": ["df_interactions", "df_content"],
    "Shape finale": [str(df_interactions.shape), str(df_content.shape)],
}
print(pd.DataFrame(recap_clean).to_string(index=False))
print("=" * 60)

---

## Cellule 4 — EDA 

### Objectif
Faire une exploration visuelle rapide de `df_interactions` pour répondre à 4 questions simples :
1. Les notes sont-elles équilibrées ou trop positives ?
2. Quels livres reçoivent le plus de notes ?
3. Combien d'utilisateurs et de livres sont en situation de cold start ?
4. La matrice user × book est-elle très creuse (sparse) ?

### Ce que cette cellule va produire
- Figure 1 : distribution des notes + Top 15 livres les plus notés
- Figure 2 : histogrammes cold start (users et books, échelle log)
- Figure 3 : heatmap d'un échantillon de la matrice
- Un résumé chiffré final (users, books, interactions, sparsité)

### Définitions utiles
- **Sparsité** = $1 - \frac{\text{interactions}}{\text{users} \times \text{books}}$
- **Cold start user** : utilisateur avec moins de 5 notes
- **Cold start book** : livre avec moins de 3 notes

In [ ]:
# =============================================================================
# CELLULE 4 — EDA 
# =============================================================================

# 1) Métriques de base
ratings_per_user = df_interactions.groupby("user_id_mapping")["rating"].count()
ratings_per_book = df_interactions.groupby("book_id_mapping")["rating"].count()

n_users = df_interactions["user_id_mapping"].nunique()
n_books = df_interactions["book_id_mapping"].nunique()
n_interactions = len(df_interactions)
sparsity = 1 - n_interactions / (n_users * n_books)

print("── Métriques globales")
print(f"Utilisateurs uniques : {n_users:,}")
print(f"Livres uniques       : {n_books:,}")
print(f"Interactions totales : {n_interactions:,}")
print(f"Sparsité             : {sparsity*100:.4f}%")
print(f"Densité              : {(1 - sparsity)*100:.4f}%")
print()

# 2) Figure 1 : Distribution des notes + Top 15 livres
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EDA — Vue globale des notes", fontsize=14, fontweight="bold")

# 2.1 Distribution des notes
rating_counts = df_interactions["rating"].value_counts().sort_index()
axes[0].bar(
    rating_counts.index.astype(str),
    rating_counts.values,
    color=sns.color_palette("muted", 5),
    edgecolor="white",
    linewidth=0.8,
 )
axes[0].set_title("4.1 — Distribution des notes")
axes[0].set_xlabel("Note")
axes[0].set_ylabel("Nombre d'interactions")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

for bar, val in zip(axes[0].patches, rating_counts.values):
    pct = val / n_interactions * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + n_interactions * 0.005,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# 2.2 Top 15 livres les plus notés
top15_books = ratings_per_book.nlargest(15).reset_index()
top15_books.columns = ["book_id_mapping", "n_ratings"]

id_to_title = (
    df_interactions[["book_id_mapping", "title"]]
    .drop_duplicates("book_id_mapping")
    .set_index("book_id_mapping")["title"]
    .astype(str)
)

def shorten_title(t, max_len=30):
    if pd.isna(t):
        return "Unknown title"
    t = str(t)
    return t if len(t) <= max_len else t[:max_len] + "…"

top15_books["title_short"] = top15_books["book_id_mapping"].map(id_to_title).apply(shorten_title)

axes[1].barh(
    top15_books["title_short"],
    top15_books["n_ratings"],
    color=sns.color_palette("muted")[0],
    edgecolor="white",
)
axes[1].invert_yaxis()
axes[1].set_title("4.2 — Top 15 livres les plus notés")
axes[1].set_xlabel("Nombre de notes reçues")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("eda_fig1_ratings_top15.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 1 sauvegardée : eda_fig1_ratings_top15.png\n")

# 3) Figure 2 : Cold Start (users/books)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EDA — Analyse du Cold Start", fontsize=14, fontweight="bold")

# 3.1 Notes par utilisateur
axes[0].hist(
    ratings_per_user.values,
    bins=50,
    color=sns.color_palette("muted")[1],
    edgecolor="white",
    log=True,
)
cold_start_users = (ratings_per_user < 5).sum()
axes[0].axvline(5, color="crimson", linestyle="--", linewidth=1.5, label="Seuil = 5")
axes[0].set_title("4.3 — Notes par utilisateur (log)")
axes[0].set_xlabel("Nombre de notes données")
axes[0].set_ylabel("Nb d'utilisateurs (log)")
axes[0].legend()
axes[0].text(
    0.97,
    0.95,
    f"Cold start users: {cold_start_users:,}\n({cold_start_users/n_users*100:.1f}% < 5 notes)",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

# 3.2 Notes par livre
axes[1].hist(
    ratings_per_book.values,
    bins=50,
    color=sns.color_palette("muted")[2],
    edgecolor="white",
    log=True,
)
cold_start_books = (ratings_per_book < 3).sum()
axes[1].axvline(3, color="crimson", linestyle="--", linewidth=1.5, label="Seuil = 3")
axes[1].set_title("4.4 — Notes par livre (log)")
axes[1].set_xlabel("Nombre de notes reçues")
axes[1].set_ylabel("Nb de livres (log)")
axes[1].legend()
axes[1].text(
    0.97,
    0.95,
    f"Cold start books: {cold_start_books:,}\n({cold_start_books/n_books*100:.1f}% < 3 notes)",
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

plt.tight_layout()
plt.savefig("eda_fig2_cold_start.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 2 sauvegardée : eda_fig2_cold_start.png\n")

# 4) Figure 3 : Heatmap densité (échantillon 50x50 pour lisibilité)
top50_users = ratings_per_user.nlargest(50).index
top50_books = ratings_per_book.nlargest(50).index

sample_matrix = (
    df_interactions[
        df_interactions["user_id_mapping"].isin(top50_users)
        & df_interactions["book_id_mapping"].isin(top50_books)
    ]
    .pivot_table(
        index="user_id_mapping",
        columns="book_id_mapping",
        values="rating",
        aggfunc="mean",
    )
    .reindex(index=top50_users, columns=top50_books)
)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    sample_matrix,
    ax=ax,
    cmap="YlOrRd",
    mask=sample_matrix.isna(),
    linewidths=0.3,
    linecolor="lightgrey",
    cbar_kws={"label": "Note moyenne"},
    xticklabels=False,
    yticklabels=False,
)

density_sample = sample_matrix.notna().sum().sum() / (50 * 50)
ax.set_title(
    "4.5 — Densité matrice user × book\n"
    f"(échantillon: {density_sample*100:.1f}% | global: {sparsity*100:.2f}% )",
    fontsize=11,
)
ax.set_xlabel("Livres")
ax.set_ylabel("Utilisateurs")

plt.tight_layout()
plt.savefig("eda_fig3_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 3 sauvegardée : eda_fig3_heatmap.png\n")

# 5) Résumé final
print("=" * 60)
print("RÉCAPITULATIF EDA")
print("=" * 60)
print(f"Utilisateurs         : {n_users:,}")
print(f"Livres               : {n_books:,}")
print(f"Interactions         : {n_interactions:,}")
print(f"Sparsité globale     : {sparsity*100:.4f}%")
print(f"Note moyenne         : {df_interactions['rating'].mean():.3f} / 5")
print(f"Note médiane         : {df_interactions['rating'].median():.1f} / 5")
print(f"Cold start users     : {cold_start_users:,} ({cold_start_users/n_users*100:.1f}%)")
print(f"Cold start books     : {cold_start_books:,} ({cold_start_books/n_books*100:.1f}%)")
print("=" * 60)

print(df_interactions['rating'].value_counts().sort_index())

# 6) Diagnostic explicite du biais de notation
low_ratings = int(rating_counts.get(1, 0) + rating_counts.get(2, 0))
neutral_ratings = int(rating_counts.get(3, 0))
high_ratings = int(rating_counts.get(4, 0) + rating_counts.get(5, 0))

low_pct = (low_ratings / n_interactions) * 100 if n_interactions else 0.0
neutral_pct = (neutral_ratings / n_interactions) * 100 if n_interactions else 0.0
high_pct = (high_ratings / n_interactions) * 100 if n_interactions else 0.0

imbalance_ratio = (high_ratings / low_ratings) if low_ratings > 0 else np.inf

print("\n" + "-" * 60)
print("DIAGNOSTIC BIAIS DE NOTATION")
print("-" * 60)
print(f"Ratings bas (1-2)     : {low_ratings:,} ({low_pct:.2f}%)")
print(f"Ratings neutres (3)   : {neutral_ratings:,} ({neutral_pct:.2f}%)")
print(f"Ratings hauts (4-5)   : {high_ratings:,} ({high_pct:.2f}%)")
print(f"Ratio hauts/bas       : {imbalance_ratio:.2f}x")

if high_pct >= 60:
    print(" Dataset biaisé vers les notes élevées (positive bias).")
else:
    print(" Biais modéré des notes.")

title_consistency = df_interactions.groupby('book_id_mapping')['title'].nunique()
print("Livres avec plusieurs titres :", (title_consistency > 1).sum())

In [ ]:
# =============================================================================
# CELLULE 4.1 — Poids de classes (anti-biais de notes)
# =============================================================================

if "df_interactions" not in globals():
    raise ValueError("df_interactions introuvable. Exécute d'abord les cellules de préparation.")

# Groupes de ratings
# low: 1-2 | mid: 3 | high: 4-5
rating_series = df_interactions["rating"]

group_labels = np.select(
    [rating_series <= 2, rating_series == 3, rating_series >= 4],
    ["low_1_2", "mid_3", "high_4_5"],
    default="unknown",
)

group_counts = pd.Series(group_labels).value_counts()

total = int(group_counts.sum())
n_groups = 3

# Poids inverses à la fréquence: w_g = total / (n_groups * count_g)
group_weights = {
    g: (total / (n_groups * int(group_counts[g])))
    for g in ["low_1_2", "mid_3", "high_4_5"]
    if g in group_counts and group_counts[g] > 0
}

# Poids par rating individuel (1..5)
rating_to_group = {1: "low_1_2", 2: "low_1_2", 3: "mid_3", 4: "high_4_5", 5: "high_4_5"}
rating_class_weights = {
    r: float(group_weights[rating_to_group[r]])
    for r in [1, 2, 3, 4, 5]
    if rating_to_group[r] in group_weights
}

# Exemple: vecteur sample_weight aligné sur df_interactions
sample_weight = rating_series.map(rating_class_weights).astype("float32")

print("=" * 64)
print("POIDS DE CLASSES — ANTI-BIAIS DE NOTES")
print("=" * 64)
print("Comptes par groupe:")
print(group_counts.reindex(["low_1_2", "mid_3", "high_4_5"]).fillna(0).astype(int).to_string())
print("\nPoids par groupe:")
print(pd.Series(group_weights).reindex(["low_1_2", "mid_3", "high_4_5"]).to_string())
print("\nPoids par rating:")
print(pd.Series(rating_class_weights).sort_index().to_string())
print(f"\nSample weight ready: shape={sample_weight.shape}, dtype={sample_weight.dtype}")
print("=" * 64)

# Utilisation type:
# - sklearn metrics: mean_squared_error(y_true, y_pred, sample_weight=sample_weight)
# - entraînement (modèles qui acceptent sample_weight): fit(X, y, sample_weight=sample_weight)

## **Audit de qualité des données (avant filtrage)**

### Validation systématique
- **Missing values** : vérifier les valeurs NULL
- **Duplicates** : détecter les doublons exacts
- **Mappings** : validation des encodages ID
- **Rating range** : vérifier la validité des notes
- **Sparsity** : calculer le taux de complétude
- **Activity filtering** : profils d'activité utilisateur/livre
- **Metadata coverage** : complétude des colonnes descriptives
- **Consistency checks** : cohérence inter-tables

In [ ]:
# =============================================================================
# CELLULE 4.5 — Audit de qualité des données (avant filtrage)
# =============================================================================

print("="*70)
print("AUDIT DE QUALITÉ — AVANT FILTRAGE")
print("="*70)
print()

# 1) Missing Values
print("1⃣  MISSING VALUES")
print("─" * 70)
missing_counts = df_interactions.isnull().sum()
missing_pct = (df_interactions.isnull().sum() / len(df_interactions) * 100)
missing_df = pd.DataFrame({
    "Colonne": missing_counts.index,
    "Count NULL": missing_counts.values,
    "% NULL": missing_pct.values
})
missing_df = missing_df[missing_df["Count NULL"] > 0] if (missing_df["Count NULL"] > 0).any() else missing_df.head(0)
print(missing_df.to_string(index=False) if len(missing_df) > 0 else " Aucune valeur manquante détectée")
print()

# 2) Duplicates
print("2⃣  DUPLICATES")
print("─" * 70)
exact_dupes = df_interactions.duplicated().sum()
user_book_dupes = df_interactions.duplicated(subset=['user_id_mapping', 'book_id_mapping']).sum()
print(f"Doublons exacts (toutes colonnes)   : {exact_dupes:,}")
print(f"Doublons (user, book) pairs         : {user_book_dupes:,}")
if user_book_dupes > 0:
    print("  Attention: Le même utilisateur a noté le même livre plusieurs fois")
print()

# 3) Mappings validity
print("3⃣  MAPPINGS VALIDITY")
print("─" * 70)
user_mapping_valid = (df_interactions['user_id_mapping'] >= 0).all()
book_mapping_valid = (df_interactions['book_id_mapping'] >= 0).all()
print(f"user_id_mapping valides (≥0)       : {user_mapping_valid}")
print(f"book_id_mapping valides (≥0)       : {book_mapping_valid}")
print(f"User ID range                      : [{df_interactions['user_id_mapping'].min()} - {df_interactions['user_id_mapping'].max()}]")
print(f"Book ID range                      : [{df_interactions['book_id_mapping'].min()} - {df_interactions['book_id_mapping'].max()}]")
print()

# 4) Rating range & distribution
print("4⃣  RATING RANGE & DISTRIBUTION")
print("─" * 70)
print(f"Min rating                         : {df_interactions['rating'].min()}")
print(f"Max rating                         : {df_interactions['rating'].max()}")
print(f"Mean rating                        : {df_interactions['rating'].mean():.2f}")
print(f"Median rating                      : {df_interactions['rating'].median():.2f}")
print(f"Std dev                            : {df_interactions['rating'].std():.2f}")
print()
print("Répartition des notes:")
rating_dist = df_interactions['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    pct = count / len(df_interactions) * 100
    print(f"  Rating {rating:g} : {count:6,} ({pct:5.2f}%)")
print()

# 5) Sparsity (avant filtrage)
print("5⃣  SPARSITY (avant filtrage)")
print("─" * 70)
n_users = df_interactions['user_id_mapping'].nunique()
n_books = df_interactions['book_id_mapping'].nunique()
n_interactions = len(df_interactions)
max_possible = n_users * n_books
sparsity_pct = (1 - n_interactions / max_possible) * 100
density_pct = (n_interactions / max_possible) * 100
print(f"Users (uniques)                    : {n_users:,}")
print(f"Books (uniques)                    : {n_books:,}")
print(f"Interactions (total)               : {n_interactions:,}")
print(f"Max possible interactions          : {max_possible:,}")
print(f"Density                            : {density_pct:.4f}%")
print(f"Sparsity                           : {sparsity_pct:.2f}%")
print()

# 6) Activity filtering (distribution d'activité)
print("6⃣  ACTIVITY FILTERING")
print("─" * 70)
user_activity = df_interactions.groupby('user_id_mapping')['rating'].count()
book_activity = df_interactions.groupby('book_id_mapping')['rating'].count()

print("Activité utilisateurs (notes par user):")
print(f"  Min                              : {user_activity.min()}")
print(f"  Max                              : {user_activity.max()}")
print(f"  Mean                             : {user_activity.mean():.1f}")
print(f"  Median                           : {user_activity.median():.1f}")
print(f"  Q1 (25%)                         : {user_activity.quantile(0.25):.1f}")
print(f"  Q3 (75%)                         : {user_activity.quantile(0.75):.1f}")
print()
print("Activité livres (notes par book):")
print(f"  Min                              : {book_activity.min()}")
print(f"  Max                              : {book_activity.max()}")
print(f"  Mean                             : {book_activity.mean():.1f}")
print(f"  Median                           : {book_activity.median():.1f}")
print(f"  Q1 (25%)                         : {book_activity.quantile(0.25):.1f}")
print(f"  Q3 (75%)                         : {book_activity.quantile(0.75):.1f}")
print()

# 7) Metadata coverage
print("7⃣  METADATA COVERAGE")
print("─" * 70)
metadata_cols = ['title', 'authors', 'author', 'description', 'genre', 'image_url']
for col in metadata_cols:
    if col in df_interactions.columns:
        non_null = df_interactions[col].notna().sum()
        coverage = non_null / len(df_interactions) * 100
        print(f"{col:20} : {non_null:6,} / {len(df_interactions):,} ({coverage:5.2f}%)")
    elif col in df_content.columns:
        non_null = df_content[col].notna().sum()
        coverage = non_null / len(df_content) * 100
        print(f"{col:20} : {non_null:6,} / {len(df_content):,} ({coverage:5.2f}%)")
print()

# 8) Consistency checks
print("8⃣  CONSISTENCY CHECKS")
print("─ " * 35)

# 8a) Title consistency
title_consistency = df_interactions.groupby('book_id_mapping')['title'].nunique()
inconsistent_books = (title_consistency > 1).sum()
print(f"Livres avec plusieurs titres       : {inconsistent_books}")
if inconsistent_books > 0:
    print(f"    {inconsistent_books} books avoir discrepancies — utiliser le titre modal")
print()

# 8b) Diagnostic: Check what columns exist where
print("DIAGNOSTIC — Colonnes disponibles:")
print("─" * 70)
print(f"df_interactions.columns ({len(df_interactions.columns)}): {df_interactions.columns.tolist()}")
print(f"\ndf_content.columns ({len(df_content.columns)}): {df_content.columns.tolist()}")
print()

# 8c) User-Book overlap — vérifier les colonnes disponibles
print("Tentative de fusion df_interactions ← df_content...")

# Identify common metadata columns (excluding join key and rating)
common_metadata = set(df_interactions.columns) & set(df_content.columns)
common_metadata.discard('book_id_mapping')
common_metadata.discard('rating')

print(f"Colonnes communes (metadata): {list(common_metadata)}")

if common_metadata:
    # Merge - les colonnes communes vont être dupliquées, pandas ajoute des suffixes
    df_merged = df_interactions.merge(
        df_content[['book_id_mapping'] + list(common_metadata)],
        on='book_id_mapping', how='left',
        suffixes=('_interactions', '_content')
    )

    print(f"Après merge, colonnes: {df_merged.columns.tolist()}")

    # Compte les interactions avec metadata complète
    metadata_cols_content = [col for col in df_merged.columns if col.endswith('_content')]
    if metadata_cols_content:
        orphan_count = df_merged[metadata_cols_content].isnull().all(axis=1).sum()
        covered_books = len(df_merged) - orphan_count
        print(f"\nInteractions avec metadata : {covered_books:,} / {len(df_merged):,}")
        print(f"  → Metadata columns: {metadata_cols_content}")
    else:
        print("  PROBLÈME: Les colonnes metadata ont disparu après la fusion")
else:
    print("  Pas de colonnes metadata communes entre df_interactions et df_content")
    print("   Cela signifie que df_interactions a déjà tout ce qu'on a besoin")
    print("   Pas besoin de faire un merge!")

    # Vérifier si les données sont déjà là
    metadata_in_interactions = [
        col for col in ['title', 'authors', 'author', 'description', 'genre']
        if col in df_interactions.columns
    ]
    if metadata_in_interactions:
        print(f"    Ces colonnes sont déjà dans df_interactions: {metadata_in_interactions}")
        print()
        print(" CONCLUSION: df_interactions contient DÉJÀ toutes les métadonnées!")
        print("   Aucun merge additionnel n'est nécessaire.")
        print("   Les données sont prêtes pour la modélisation.")
    else:
        print("  Avertissement: Pas de metadata détectée du tout!")

print()
print("="*70)


## **Détection des livres dupliqués **

### Problème
Même livre peut apparaître avec **plusieurs IDs** à cause de:
- Variations dans les titres (accents, espaces, troncatures)
- Différentes éditions ("Hardcover", "Paperback", "eBook")
- Erreurs de saisie dans les données sources

Impact: **recommandations fragmentées** pour le même livre

In [ ]:
import pandas as pd
from difflib import SequenceMatcher

# ==========================================
# 4.6: AUDIT DES DOUBLONS DE LIVRES
# ==========================================
print("\n" + "=" * 70)
print("4.6 - AUDIT DES DOUBLONS DE LIVRES (Exact + Semantic)")
print("=" * 70)

# Initialize defaults for all variables
exact_dupes = 0
potential_dupes_df = pd.DataFrame()
total_books = 0
unique_titles = 0
title_author_unique = 0

# 1) Doublons exacts sur (title, authors/author/name)
print("1⃣  DOUBLONS EXACTS (même titre + même auteur)")
print("─" * 70)

# Find the author column (might be 'authors', 'author', or 'name')
author_col = None
for col in ['authors', 'author', 'name']:
    if col in df_content.columns:
        author_col = col
        break

if author_col and 'title' in df_content.columns:
    book_meta = df_content[['book_id_mapping', 'title', author_col]].copy()
    book_meta.rename(columns={author_col: 'author'}, inplace=True)
    exact_dupes = book_meta.duplicated(subset=['title', 'author'], keep=False).sum()
    print(f"Livres avec titre ET auteur ({author_col}) identiques : {exact_dupes}")
    
    if exact_dupes > 0:
        dup_groups = book_meta[book_meta.duplicated(subset=['title', 'author'], keep=False)].sort_values('title')
        print("\nExemples de doublons exacts:")
        for title in dup_groups['title'].unique()[:5]:
            ids = dup_groups[dup_groups['title'] == title]['book_id_mapping'].tolist()
            print(f"  {title[:60]} → IDs: {ids}")
else:
    print(f"Colonnes disponibles dans df_content: {df_content.columns.tolist()}")
    print("  Impossible de faire la comparaison: 'title' ou colonne auteur manquante")
    book_meta = None
print()

# 2) Doublons potentiels (titre très similaire)
print("2⃣  DOUBLONS POTENTIELS (titres similaires > 85%)")
print("─" * 70)

def string_similarity(a, b):
    """Calcule similarité entre 2 strings (0-1)"""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

potential_dupes = []
if book_meta is not None:
    book_titles = list(book_meta['title'].dropna().unique())

    # Vérifier les titres similaires (> 85% match)
    for i, title1 in enumerate(book_titles):
        for title2 in book_titles[i+1:]:
            sim = string_similarity(title1, title2)
            if sim > 0.85 and sim < 1.0:  # 85-99% de similarité = suspect
                potential_dupes.append({
                    'title1': title1,
                    'title2': title2,
                    'similarity': sim
                })

    potential_dupes_df = pd.DataFrame(potential_dupes).sort_values('similarity', ascending=False) if potential_dupes else pd.DataFrame()

    print(f"Pairs de titres similaires (85%-99%) : {len(potential_dupes_df)}")
    if len(potential_dupes_df) > 0:
        print("\nTop 15 doublons potentiels:")
        for idx, row in potential_dupes_df.head(15).iterrows():
            ids1 = book_meta[book_meta['title'] == row['title1']]['book_id_mapping'].tolist()
            ids2 = book_meta[book_meta['title'] == row['title2']]['book_id_mapping'].tolist()
            print(f"  {row['similarity']:.1%} | {row['title1'][:40]:40s} vs {row['title2'][:40]:40s}")
            print(f"         → IDs {ids1} vs {ids2}")
    else:
        print(" Pas de titres similaires détectés")
else:
    print("  Impossible de calculer: données manquantes")
print()

# 3) Analyse des doublons exacts par nombre de notes
print("3⃣  IMPACT DES DOUBLONS — Fragmentation des notes")
print("─" * 70)

if book_meta is not None and author_col and exact_dupes > 0:
    # Pour chaque groupe de livres dupliqués, calculer l'impact
    dup_titles = book_meta[book_meta.duplicated(subset=['title', 'author'], keep=False)]['title'].unique()
    
    fragmentation_impact = []
    for title in dup_titles[:10]:  # Top 10 pour l'affichage
        book_ids = book_meta[book_meta['title'] == title]['book_id_mapping'].tolist()
        
        # Compter les notes pour chaque ID
        rating_counts = []
        total_ratings = 0
        for bid in book_ids:
            count = (df_interactions['book_id_mapping'] == bid).sum()
            rating_counts.append(count)
            total_ratings += count
        
        if total_ratings > 0:
            fragmentation_impact.append({
                'title': title,
                'num_ids': len(book_ids),
                'book_ids': book_ids,
                'rating_counts': rating_counts,
                'total_ratings': total_ratings
            })
    
    if fragmentation_impact:
        print(f"\nFragmentation détectée (notes dispersées):")
        for item in sorted(fragmentation_impact, key=lambda x: x['total_ratings'], reverse=True):
            print(f"  Titre: {item['title'][:60]}")
            for bid, cnt in zip(item['book_ids'], item['rating_counts']):
                print(f"    → ID {bid}: {cnt:,} notes")
            print(f"      TOTAL: {item['total_ratings']:,} notes sur {item['num_ids']} IDs (fragmentation)")
            print()
else:
    print(" Pas de doublons exacts détectés → pas de fragmentation")

print()

# 4) Résumé et recommandations
print("4⃣  RÉSUMÉ & RECOMMANDATIONS")
print("─" * 70)

if book_meta is not None:
    total_books = book_meta['book_id_mapping'].nunique()
    unique_titles = book_meta['title'].nunique()
    title_author_unique = book_meta.drop_duplicates(subset=['title', 'author']).shape[0]

    print(f"Total books (IDs)                      : {total_books:,}")
    print(f"Unique titles                          : {unique_titles:,}")
    print(f"Unique (title, author) pairs           : {title_author_unique:,}")
    print(f"Exact duplicates (title+author)        : {exact_dupes:,}")
    print(f"Potential semantic duplicates          : {len(potential_dupes_df):,}")
    print()

    print("RECOMMANDATIONS:")
    if exact_dupes > 0:
        print("  ACTION REQUIRED: Doublons exacts détectés!")
        print("   Solution: Fusionner par (title, author) → consolider les ratings")
        print("   Impact: Pourrait augmenter le poids de ces livres populaires")
    else:
        print(" Pas de doublons exacts → Système OK")

    if len(potential_dupes_df) > 0:
        print(f"\n  À VÉRIFIER: {len(potential_dupes_df)} paires de titres similaires")
        print("   Actions: Revue manuelle + fuzzy matching + Levenshtein distance")
    else:
        print("\n Pas de titres similaires → Fondation solide")
        
    print("\nConclusion: Système de doublons = ", end="")
    if exact_dupes > 0 or len(potential_dupes_df) > 5:
        print("  À améliorer (voir actions ci-dessus)")
    else:
        print(" ROBUSTE")
else:
    print("  Impossible de générer le résumé: données manquantes")
    print("Vérifiez que 'title' et une colonne auteur existent dans df_content")

print()



## Cellule 6 — Chargement des données en mémoire & Modèle Baseline

### Objectif
Utiliser directement les tables préparées en Cellule 5 (`df_filtered`, `df_content_filtered`) et construire le **modèle Baseline** : `get_popular_recommendations()`.

### Pourquoi un Baseline ?
Le Baseline est le **point de comparaison minimal** : tout modèle CF, Content-Based ou Hybride doit faire mieux que lui pour justifier sa complexité.

### Stratégie du Baseline
Score de popularité pondérée (inspiré IMDb) :

$$\text{score}(b) = \frac{v}{v + m} \times \bar{R}_b + \frac{m}{v + m} \times \bar{R}_{global}$$

### Sanity Checks
- Vérifier que `df_filtered` et `df_content_filtered` existent
- Vérifier la continuité de `user_idx` et `book_idx`
- Tester `get_popular_recommendations()`

In [ ]:
# =============================================================================
# CELLULE 1 — Chargement en mémoire & Modèle Baseline
# =============================================================================

# ── 1.1  Préconditions (issues de la Cellule 5) ───────────────────────────────
required_tables = ["df_filtered", "df_content_filtered"]
missing_tables = [name for name in required_tables if name not in globals()]

if missing_tables:
    print(f" Tables manquantes depuis la Cellule 5: {missing_tables}")
    print("   Tentative de fallback depuis df_interactions / df_content...")

    if "df_interactions" in globals() and "df_content" in globals():
        df_filtered = df_interactions.copy()
        df_content_filtered = df_content.copy()

        # Recréer les index continus si absents
        if "user_idx" not in df_filtered.columns or "book_idx" not in df_filtered.columns:
            unique_users = np.sort(df_filtered["user_id_mapping"].unique())
            unique_books = np.sort(df_filtered["book_id_mapping"].unique())
            user_old_to_new = {old: new for new, old in enumerate(unique_users)}
            book_old_to_new = {old: new for new, old in enumerate(unique_books)}

            df_filtered["user_idx"] = df_filtered["user_id_mapping"].map(user_old_to_new).astype("int32")
            df_filtered["book_idx"] = df_filtered["book_id_mapping"].map(book_old_to_new).astype("int32")

        if "book_idx" not in df_content_filtered.columns:
            book_map = (
                df_filtered[["book_id_mapping", "book_idx"]]
                .drop_duplicates("book_id_mapping")
                .set_index("book_id_mapping")["book_idx"]
            )
            df_content_filtered["book_idx"] = df_content_filtered["book_id_mapping"].map(book_map).astype("Int32")

        print(" Fallback activé: df_filtered et df_content_filtered reconstruits")
    else:
        raise ValueError(
            "Exécute d'abord la Cellule 5 (ou au minimum les cellules qui créent df_interactions et df_content)."
        )

# Copie de travail pour la partie modélisation
df = df_filtered.copy()
df_content = df_content_filtered.copy()

print(" Données récupérées en mémoire")
print(f"   df (interactions) : {df.shape}")
print(f"   df_content        : {df_content.shape}")

# ── 1.2  Vérification rapide post-chargement ─────────────────────────────────
n_users = df["user_idx"].nunique()
n_books = df["book_idx"].nunique()
sparsity = 1 - len(df) / (n_users * n_books)

user_idx_min = int(df["user_idx"].min())
user_idx_max = int(df["user_idx"].max())
book_idx_min = int(df["book_idx"].min())
book_idx_max = int(df["book_idx"].max())
ratings_in_range = bool(df["rating"].between(1, 5).all())
duplicate_pairs = int(df.duplicated(subset=["user_idx", "book_idx"]).sum())

print(
    "Verification rapide post-chargement: "
    f"user_idx [{user_idx_min}..{user_idx_max}] | "
    f"book_idx [{book_idx_min}..{book_idx_max}] | "
    f"ratings OK={ratings_in_range} | "
    f"doublons={duplicate_pairs}"
)
print(f"   {n_users:,} utilisateurs | {n_books:,} livres | {len(df):,} interactions")
print(f"   Sparsité : {sparsity*100:.4f}%")

# ── 1.3  Lookups globaux ───────────────────────────────────────────────────────
idx_to_title: dict = (
    df[["book_idx", "title"]]
    .drop_duplicates("book_idx")
    .set_index("book_idx")["title"]
    .astype(str)
    .to_dict()
)

title_to_idx: dict = {
    str(title).lower().strip(): idx
    for idx, title in idx_to_title.items()
}

idx_to_book_mapping: dict = (
    df[["book_idx", "book_id_mapping"]]
    .drop_duplicates("book_idx")
    .set_index("book_idx")["book_id_mapping"]
    .to_dict()
)

print(f"\n Lookups construits : {len(idx_to_title):,} livres indexés")

BOOK_METADATA_LOOKUP = (
    df_content[["book_idx", "author", "genre", "image_url", "url"]]
    .dropna(subset=["book_idx"])
    .drop_duplicates("book_idx")
    .copy()
)
BOOK_METADATA_LOOKUP["book_idx"] = pd.to_numeric(BOOK_METADATA_LOOKUP["book_idx"], errors="coerce").astype("Int64")



# Fonctions attach_book_metadata et display_recommendation_gallery supprimees.
# Elles ne sont plus utilisees (remplacees par des impressions simples).

CONFIDENCE_PERCENTILE = 60

def get_popular_recommendations(
    top_n: int = 10,
    exclude_book_idxs: list = None,
    min_confidence_percentile: int = CONFIDENCE_PERCENTILE,
) -> pd.DataFrame:
    if exclude_book_idxs is None:
        exclude_book_idxs = []

    book_stats = (
        df.groupby("book_idx")["rating"]
        .agg(avg_rating="mean", n_ratings="count")
        .reset_index()
    )

    m = np.percentile(book_stats["n_ratings"], min_confidence_percentile)
    r_global = df["rating"].mean()

    v = book_stats["n_ratings"]
    r = book_stats["avg_rating"]
    book_stats["score"] = (v / (v + m)) * r + (m / (v + m)) * r_global

    if exclude_book_idxs:
        book_stats = book_stats[~book_stats["book_idx"].isin(exclude_book_idxs)]

    top_books = book_stats.nlargest(top_n, "score").copy()
    top_books["title"] = top_books["book_idx"].map(idx_to_title)

    return top_books[["book_idx", "title", "avg_rating", "n_ratings", "score"]].reset_index(drop=True)

# ── 1.5  Vérification rapide ─────────────────────────────────────────────────
print("\n── Test : get_popular_recommendations(top_n=10)")
baseline_recs = get_popular_recommendations(top_n=10)
print(baseline_recs.to_string(index=False))

print("\n── Test avec exclusion des 3 premiers livres")
exclude = baseline_recs["book_idx"].head(3).tolist()
baseline_recs_exc = get_popular_recommendations(top_n=10, exclude_book_idxs=exclude)
overlap = set(baseline_recs_exc["book_idx"]) & set(exclude)
print(f"Verification rapide exclusion: {len(overlap)} livre(s) exclus retrouvés")

# ── 1.6  Visualisation ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("muted", len(baseline_recs))
bars = ax.barh(
    baseline_recs["title"].str.slice(0, 35).str.cat(["…"] * len(baseline_recs)),
    baseline_recs["score"],
    color=colors,
    edgecolor="white",
)
ax.invert_yaxis()
ax.set_xlabel("Score de popularité pondérée (IMDb)")
ax.set_title("Baseline — Top 10 livres les plus populaires", fontweight="bold")
for bar, row in zip(bars, baseline_recs.itertuples()):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f" {row.avg_rating:.2f}  ({row.n_ratings} votes)",
        va="center", fontsize=8.5,
    )
ax.set_xlim(0, baseline_recs["score"].max() * 1.25)
plt.tight_layout()
plt.savefig("baseline_top10.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n Figure sauvegardée : baseline_top10.png")

In [ ]:
# =============================================================================
# CELLULE 7 — Content-Based : Corpus TF-IDF & Similarité Cosinus [FIXED]
# =============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── 2.1  Construction du corpus textuel ───────────────────────────────────────
def build_corpus(row) -> str:
    title_str  = str(row.get("title",  "")).strip()
    author_str = str(row.get("author", "")).strip()
    genre_str  = str(row.get("genre",  "")).strip()
    desc_str   = str(row.get("description", "")).strip()

    parts = (
        [title_str]  * 3 +
        [author_str] * 3 +
        [genre_str]  * 2 +
        [desc_str]
    )
    return " ".join(p for p in parts if p and p.lower() not in ("nan", "unknown", ""))

df_content["corpus"] = df_content.apply(build_corpus, axis=1)

n_empty_corpus = (df_content["corpus"].str.strip() == "").sum()
print(f"── Corpus construit : {len(df_content):,} livres")
print(f"   Corpus vides : {n_empty_corpus}")
print()

# ──2.2  Vectorisation TF-IDF ─────────────────────────────────────────────────
print("── Vectorisation TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(
    max_features   = 8_000,
    ngram_range    = (1, 2),
    stop_words     = "english",
    sublinear_tf   = True,
    min_df         = 2,
    dtype          = np.float32,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df_content["corpus"])
print(
    "Verification rapide TF-IDF: "
    f"sparse={sparse.issparse(tfidf_matrix)} | "
    f"lignes matrice={tfidf_matrix.shape[0]} | "
    f"livres content={len(df_content)}"
)
print(f"   Matrice TF-IDF : {tfidf_matrix.shape}")
print()

# ── 2.3  Index book_idx → ligne dans df_content ───────────────────────────────
df_content_reset = df_content.reset_index(drop=True)

# Construction du mapping en ignorant les lignes où book_idx est NA
book_mapping_to_idx_in_content = {}
book_idx_to_row = {}

for row_idx, row in df_content_reset.iterrows():
    bidx = row.get("book_idx")
    bm = row.get("book_id_mapping")

    if pd.isna(bidx) or pd.isna(bm):
        continue

    bidx = int(bidx)
    bm = int(bm)

    book_mapping_to_idx_in_content[bm] = bidx
    book_idx_to_row[bidx] = row_idx

print(f"── Index construction FROM df_content:")
print(f"   book_mapping_to_idx_in_content: {len(book_mapping_to_idx_in_content)} livres")
print(f"   book_idx_to_row: {len(book_idx_to_row)} livres")
print()

# ── 2.4  Fonction get_content_recommendations() ───────────────────────────────
def get_content_recommendations(
    title_query: str,
    top_n: int = 10,
    exclude_same_author: bool = False,
) -> pd.DataFrame:
    """Recommande les livres les plus similaires à `title_query` via cosinus TF-IDF."""

    query_lower = title_query.lower().strip()
    if query_lower not in title_to_idx:
        candidates = [t for t in title_to_idx if query_lower in t]
        if not candidates:
            raise ValueError(f"Titre introuvable : '{title_query}'.")
        query_lower = candidates[0]
        print(f"  ℹ  Titre exact non trouvé → correspondance partielle : '{query_lower}'")

    ref_book_idx = title_to_idx[query_lower]

    # Vérifier que ce livre est dans df_content
    if ref_book_idx not in book_idx_to_row:
        raise ValueError(
            f"Le livre '{title_query}' (book_idx={ref_book_idx}) "
            f"n'a pas de métadonnées dans df_content."
        )

    ref_row    = book_idx_to_row[ref_book_idx]
    ref_vector = tfidf_matrix[ref_row]

    sim_scores = cosine_similarity(ref_vector, tfidf_matrix).flatten()

    result = df_content_reset.copy()
    result["similarity_score"] = sim_scores.astype(np.float32)

    result = result[result.index != ref_row]

    if exclude_same_author:
        ref_author = df_content_reset.loc[ref_row, "author"]
        if ref_author and ref_author.lower() != "unknown":
            result = result[result["author"].str.lower() != ref_author.lower()]

    top = result.nlargest(top_n, "similarity_score").copy()
    top["book_idx"] = top["book_id_mapping"].map(book_mapping_to_idx_in_content)

    return top[["book_idx", "title", "author", "genre", "similarity_score"]].reset_index(drop=True)


# ── 2.5  Vérification rapide ─────────────────────────────────────────────────
print("── Vérification rapide de get_content_recommendations()")

# Trouver un livre du baseline qui a des métadonnées dans df_content
ref_book_idx = None
for bidx in baseline_recs["book_idx"]:
    if bidx in book_idx_to_row:
        ref_book_idx = bidx
        break

if ref_book_idx is None:
    available_book_idxs = sorted(book_idx_to_row.keys())
    if not available_book_idxs:
        raise ValueError("Aucun livre exploitable dans df_content (book_idx tous manquants).")
    ref_book_idx = available_book_idxs[0]
    print("    Aucun livre du baseline n'a de métadonnées")
    print(f"      Substituant avec book_idx={ref_book_idx}")

ref_title = idx_to_title.get(ref_book_idx, f"Book {ref_book_idx}")
print(f"\n  Livre de référence : '{ref_title}'")
cb_recs = get_content_recommendations(ref_title, top_n=5)
print(cb_recs[["title", "author", "similarity_score"]].to_string(index=False))

In [ ]:
# =============================================================================
# CELLULE 7.1 — Clustering des livres (K-Means + Hierarchical) & reco par cluster
# =============================================================================
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD

# ── 7.1  Préparation des embeddings pour clustering ───────────────────────────
if "tfidf_matrix" not in globals() or "df_content_reset" not in globals():
    raise ValueError("Exécute d'abord la cellule Content-Based (TF-IDF).")

n_content_books = tfidf_matrix.shape[0]
if n_content_books < 3:
    raise ValueError("Pas assez de livres pour faire un clustering fiable.")

max_dim = min(32, tfidf_matrix.shape[1] - 1, n_content_books - 1)
cluster_dim = max(2, max_dim)

cluster_svd = TruncatedSVD(n_components=cluster_dim, random_state=42)
book_embeddings = cluster_svd.fit_transform(tfidf_matrix)

# Heuristique simple pour le nombre de clusters
n_clusters = int(np.clip(np.sqrt(n_content_books), 4, 12))

# ── 7.2  K-Means & Hierarchical ───────────────────────────────────────────────
kmeans_model = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
df_content_reset["kmeans_cluster"] = kmeans_model.fit_predict(book_embeddings)

hclust_model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
df_content_reset["hclust_cluster"] = hclust_model.fit_predict(book_embeddings)

# Ajouter les clusters dans df_content (si possible)
if "book_idx" in df_content.columns:
    cluster_map = (
        df_content_reset.loc[df_content_reset["book_idx"].notna(), ["book_idx", "kmeans_cluster", "hclust_cluster"]]
        .drop_duplicates("book_idx")
        .copy()
    )
    cluster_map["book_idx"] = cluster_map["book_idx"].astype(int)
    df_content = df_content.drop(columns=[c for c in ["kmeans_cluster", "hclust_cluster"] if c in df_content.columns])
    df_content = df_content.merge(cluster_map, on="book_idx", how="left")

# Dictionnaires d'accès rapide
book_idx_to_kmeans_cluster = {}
cluster_to_book_idxs = {}

for _, row in df_content_reset.iterrows():
    bidx = row.get("book_idx")
    cl = row.get("kmeans_cluster")
    if pd.isna(bidx) or pd.isna(cl):
        continue
    bidx = int(bidx)
    cl = int(cl)
    book_idx_to_kmeans_cluster[bidx] = cl
    cluster_to_book_idxs.setdefault(cl, set()).add(bidx)

print("=" * 70)
print("CLUSTERING LIVRES — RÉSUMÉ")
print("=" * 70)
print(f"Livres clusterisables : {n_content_books}")
print(f"Dimension embedding   : {cluster_dim}")
print(f"Nombre de clusters    : {n_clusters}")
print("\nTailles des clusters (K-Means):")
print(df_content_reset["kmeans_cluster"].value_counts().sort_index().to_string())
print("=" * 70)

# ── 7.3  Recommandation restreinte au cluster ─────────────────────────────────
def get_cluster_based_recommendations(title_query: str, top_n: int = 10) -> pd.DataFrame:
    """Recommande dans le même cluster K-Means que le livre de référence."""
    query_lower = title_query.lower().strip()
    if query_lower not in title_to_idx:
        candidates = [t for t in title_to_idx if query_lower in t]
        if not candidates:
            raise ValueError(f"Titre introuvable: '{title_query}'")
        query_lower = candidates[0]

    ref_book_idx = title_to_idx[query_lower]

    # Fallback si le titre choisi n'a pas de metadata exploitable
    if ref_book_idx not in book_idx_to_row:
        alt_idxs = [idx for _, idx in title_to_idx.items() if idx in book_idx_to_row]
        if not alt_idxs:
            raise ValueError("Aucun livre avec metadata content pour la recommandation cluster.")
        ref_book_idx = alt_idxs[0]

    if ref_book_idx not in book_idx_to_kmeans_cluster:
        raise ValueError("Livre de référence non assigné à un cluster.")

    ref_row = book_idx_to_row[ref_book_idx]
    ref_cluster = book_idx_to_kmeans_cluster[ref_book_idx]

    sim_scores = cosine_similarity(tfidf_matrix[ref_row], tfidf_matrix).flatten()

    candidate_book_idxs = cluster_to_book_idxs.get(ref_cluster, set())
    candidate_rows = [book_idx_to_row[b] for b in candidate_book_idxs if b in book_idx_to_row and b != ref_book_idx]

    # Fallback si cluster trop petit
    if len(candidate_rows) < max(3, top_n // 2):
        all_rows = np.argsort(-sim_scores)
        candidate_rows = [int(r) for r in all_rows if int(r) != ref_row][: max(top_n * 5, 20)]

    result = df_content_reset.iloc[candidate_rows].copy()
    result["similarity_score"] = sim_scores[candidate_rows]
    result["book_idx"] = result["book_idx"].astype("Int64")

    top = result.nlargest(top_n, "similarity_score").copy()
    return top[["book_idx", "title", "author", "kmeans_cluster", "similarity_score"]].reset_index(drop=True)

# ── 7.4  Nouveau livre -> embedding -> cluster -> reco cluster ────────────────
def assign_new_book_to_cluster(title: str = "", author: str = "", genre: str = "", description: str = "") -> int:
    """Assigne un nouveau livre au cluster K-Means via embedding TF-IDF + SVD."""
    text_parts = [title] * 3 + [author] * 3 + [genre] * 2 + [description]
    corpus = " ".join([str(x).strip() for x in text_parts if str(x).strip() and str(x).lower() != "nan"])
    if not corpus:
        corpus = "unknown book"

    vec = tfidf_vectorizer.transform([corpus])
    emb = cluster_svd.transform(vec)
    cluster_id = int(kmeans_model.predict(emb)[0])
    return cluster_id


def recommend_for_new_book(
    title: str = "",
    author: str = "",
    genre: str = "",
    description: str = "",
    top_n: int = 10,
) -> pd.DataFrame:
    """Recommande des livres du cluster estimé pour un nouveau livre."""
    text_parts = [title] * 3 + [author] * 3 + [genre] * 2 + [description]
    corpus = " ".join([str(x).strip() for x in text_parts if str(x).strip() and str(x).lower() != "nan"])
    if not corpus:
        corpus = "unknown book"

    vec = tfidf_vectorizer.transform([corpus])
    emb = cluster_svd.transform(vec)
    cluster_id = int(kmeans_model.predict(emb)[0])

    sim_scores = cosine_similarity(vec, tfidf_matrix).flatten()
    cluster_rows = df_content_reset.index[df_content_reset["kmeans_cluster"] == cluster_id].tolist()

    result = df_content_reset.iloc[cluster_rows].copy()
    result["similarity_score"] = sim_scores[cluster_rows]
    result = result.nlargest(top_n, "similarity_score").copy()

    return result[["title", "author", "kmeans_cluster", "similarity_score"]].reset_index(drop=True)

# ── 7.5  Vérification rapide ───────────────────────────────────────────────────
print("\n── Test cluster-based recommendation")
ref_book_idx_for_cluster = next(iter(book_idx_to_row.keys()))
ref_title_for_cluster = idx_to_title.get(ref_book_idx_for_cluster, f"Book {ref_book_idx_for_cluster}")
cluster_recs = get_cluster_based_recommendations(ref_title_for_cluster, top_n=5)
print(f"Référence: {ref_title_for_cluster}")
print(cluster_recs.to_string(index=False))

print("\n── Test nouveau livre -> cluster")
new_cluster = assign_new_book_to_cluster(
    title="New Fantasy Adventure",
    author="Unknown Author",
    genre="fantasy adventure",
    description="A young hero discovers hidden powers and a forgotten kingdom.",
)
print(f"Cluster estimé pour nouveau livre: {new_cluster}")

new_book_recs = recommend_for_new_book(
    title="New Fantasy Adventure",
    author="Unknown Author",
    genre="fantasy adventure",
    description="A young hero discovers hidden powers and a forgotten kingdom.",
    top_n=5,
)
print(new_book_recs.to_string(index=False))

---

## Cellule 8 — Collaborative Filtering : Item-Based KNN

### Objectif
Construire un moteur de recommandation **Collaborative Filtering (CF)** : à partir de la matrice user-item des notes réelles, identifier les livres similaires en fonction du comportement collectif des utilisateurs.

### Pipeline technique

```
Interactions (user_idx, book_idx, rating)
    ↓
Sparse matrix M : users × books (ratings)
    ↓
Transpose M^T : books × users
    ↓
NearestNeighbors (n_neighbors=20, metric=cosine)
    ↓
get_collaborative_recommendations(book_idx) → Top-N items similaires
```

**Choix de conception :**
| Étape | Paramètre | Justification |
|---|---|---|
| Matrice sparse | scipy.sparse.csr_matrix | Efficacité mémoire (sparsité ~99.9%) |
| Métrique distance | Cosinus | Robuste aux différences d'échelle de notes |
| Neighbors | 20 | Balance entre diversité et similarité |
| Similarité | À la demande (vectorisée) | Évite de précalculer la matrice N×N |

### Vérification rapide
- forme de la matrice d'interactions
- exemple de voisins pour un livre
- test simple sur un index hors plage

---

## Cellule 9 — Matrix Factorization : SVD pour Gap-Filling

### Objectif
Implémenter une **décomposition en facteurs latents** (SVD) pour prédire les notes manquantes dans la matrice sparse user-item. Ces prédictions serviront au recommandeur « Predictive CF ».

### Pipeline technique

```
Interactions (sparse, user_idx × book_idx)
    ↓
Center les notes (soustraire la moyenne globale + biais utilisateur + biais livre)
    ↓
TruncatedSVD (n_components=50, fit sur la matrice centrée)
    ↓
Reconstructed Matrix = U @ Σ @ V^T + biases + centre
    ↓
get_mf_predictions(user_idx, top_n) → Top-N livres par score prédit
```

**Choix de conception :**
| Paramètre | Valeur | Justification |
|---|---|---|
| n_components | 50 | Réduit la dimensionalité (50/900 ≈ 5.5% complexity) |
| Centering | Biais + centre global | Stabilise la SVD sur matrice sparse |
| Métrique | RMSE sur hold-out test | Évalue la qualité des prédictions |

### Vérification rapide
- dimensions des matrices latentes
- plage finale des notes prédites
- exemple de recommandations utilisateur

---

## Cellule 10 — Hybrid Scorer : Fusion Pondérée des 3 Moteurs

### Objectif
Combiner les scores des **3 moteurs** (Content-Based, Collaborative Filtering, Matrix Factorization) en un **score hybride unique** via une fusion pondérée normalisée.

### Pipeline technique

```
Requête utilisateur (user_idx) + livre de référence (title_query)
        ↓
┌───────────────────┬──────────────────────┬─────────────────────┐
│  Content-Based    │  Collaborative (KNN) │  Matrix Factor. SVD │
│  TF-IDF cosinus   │  item-based cosinus  │  rating prédit      │
│  [0, 1]           │  [0, 1]              │  [1, 5] → [0, 1]    │
└───────────────────┴──────────────────────┴─────────────────────┘
        ↓ normalisation Min-Max par moteur
        ↓ fusion pondérée : w_cb × cb + w_cf × cf + w_mf × mf
        ↓ tri par score hybride décroissant
        Top-N recommandations avec scores détaillés
```

**Poids par défaut :**
| Moteur | Poids | Justification |
|---|---|---|
| CB (TF-IDF) | 0.30 | Seuls 95 livres ont des métadonnées |
| CF (KNN) | 0.40 | Meilleure couverture (898 livres) |
| MF (SVD) | 0.30 | Complète les gaps de sparsité |

### Vérification rapide
- somme des poids affichée
- exemple hybride avec et sans titre de référence
- contrôle simple du score final et des livres déjà lus

In [ ]:
# =============================================================================
# CELLULE 10 — Hybrid Scorer : Fusion Pondérée CB + CF + MF
# =============================================================================

# ── 5.1  Poids de fusion ─────────────────────────────────────────────────────

W_CB = 0.30   # Content-Based (TF-IDF)
W_CF = 0.40   # Collaborative Filtering (KNN)
W_MF = 0.30   # Matrix Factorization (SVD)
weights_sum = W_CB + W_CF + W_MF

print(f"── Poids de fusion : CB={W_CB} | CF={W_CF} | MF={W_MF} | somme={weights_sum:.2f}")

# ── 5.2  Utilitaire : Normalisation Min-Max ───────────────────────────────────
def minmax_normalize(scores: np.ndarray) -> np.ndarray:
    """Normalise un vecteur de scores dans [0, 1]."""
    s_min, s_max = scores.min(), scores.max()
    if s_max - s_min < 1e-9:
        return np.zeros_like(scores)
    return (scores - s_min) / (s_max - s_min)

# ── 5.3  Fonction principale : get_hybrid_recommendations() ─────────────────
def get_hybrid_recommendations(
    user_idx: int,
    title_query: str = None,
    top_n: int = 10,
    w_cb: float = W_CB,
    w_cf: float = W_CF,
    w_mf: float = W_MF,
) -> pd.DataFrame:
    """
    Recommandations hybrides en combinant CB, CF et MF.

    Paramètres
    ----------
    user_idx   : int    — Indice utilisateur (0-based, pour MF)
    title_query: str    — Titre de référence pour CB et CF (optionnel)
    top_n      : int    — Nombre de recommandations
    w_cb/cf/mf : float  — Poids (doivent sommer à 1.0)

    Retour
    ------
    pd.DataFrame : [book_idx, title, score_cb, score_cf, score_mf, hybrid_score]
    """
    scores_cb = np.zeros(n_books)
    scores_cf = np.zeros(n_books)

    # ── MF : prédictions pour l'utilisateur (tous les livres) ────────────────
    if user_idx < 0 or user_idx >= n_users:
        raise ValueError(f"user_idx {user_idx} hors plage [0, {n_users-1}]")

    mf_raw = predicted_ratings[user_idx].copy()  # (n_books,)
    scores_mf = minmax_normalize(mf_raw)

    # ── CB : similarité cosinus depuis le livre de référence ─────────────────
    ref_book_idx_hybrid = None

    if title_query is not None:
        query_lower = title_query.lower().strip()
        if query_lower not in title_to_idx:
            candidates = [t for t in title_to_idx if query_lower in t]
            if candidates:
                query_lower = candidates[0]

        if query_lower in title_to_idx:
            ref_book_idx_hybrid = title_to_idx[query_lower]
            if ref_book_idx_hybrid in book_idx_to_row:
                ref_row = book_idx_to_row[ref_book_idx_hybrid]
                ref_vec = tfidf_matrix[ref_row]
                sim_all = cosine_similarity(ref_vec, tfidf_matrix).flatten()

                # NA-safe: ignorer les lignes où book_idx manque dans df_content_reset
                for i, row_data in df_content_reset.iterrows():
                    bidx = row_data.get("book_idx")
                    if pd.isna(bidx):
                        continue
                    bidx = int(bidx)
                    if 0 <= bidx < n_books:
                        scores_cb[bidx] = float(sim_all[i])

    # ── CF : similarité KNN depuis le livre de référence ─────────────────────
    if ref_book_idx_hybrid is not None and ref_book_idx_hybrid < n_books:
        distances, indices = knn_model.kneighbors(
            interactions_matrix_T[ref_book_idx_hybrid], n_neighbors=min(N_NEIGHBORS + 1, n_books)
        )
        distances = distances.flatten()
        indices = indices.flatten()
        similarities = 1.0 - distances
        for sim, neighbor_idx in zip(similarities, indices):
            if neighbor_idx < n_books:
                scores_cf[neighbor_idx] = float(sim)
    else:
        book_counts = np.bincount(df["book_idx"].values, minlength=n_books).astype(float)
        scores_cf = minmax_normalize(book_counts)

    # ── Normalisation Min-Max ─────────────────────────────────────────────────
    scores_cb_norm = minmax_normalize(scores_cb)
    scores_cf_norm = minmax_normalize(scores_cf)
    scores_mf_norm = scores_mf

    # ── Fusion pondérée ───────────────────────────────────────────────────────
    hybrid_scores = w_cb * scores_cb_norm + w_cf * scores_cf_norm + w_mf * scores_mf_norm

    # ── Exclure le livre de référence et les livres déjà lus ─────────────────
    books_read = set(interactions_matrix[user_idx].nonzero()[1])
    if ref_book_idx_hybrid is not None:
        books_read.add(ref_book_idx_hybrid)

    for bidx in books_read:
        if bidx < n_books:
            hybrid_scores[bidx] = -np.inf

    # ── Sélectionner Top-N ────────────────────────────────────────────────────
    top_indices = np.argsort(-hybrid_scores)[:top_n + len(books_read)]
    results = []
    for bidx in top_indices:
        if hybrid_scores[bidx] == -np.inf:
            continue
        results.append({
            "book_idx": int(bidx),
            "title": idx_to_title.get(int(bidx), "Unknown"),
            "score_cb": float(scores_cb_norm[bidx]),
            "score_cf": float(scores_cf_norm[bidx]),
            "score_mf": float(scores_mf_norm[bidx]),
            "hybrid_score": float(hybrid_scores[bidx]),
        })
        if len(results) == top_n:
            break

    return pd.DataFrame(results)


# ── 5.4  Vérification rapide ────────────────────────────────────────────────
print("\n" + "="*70)
print("Test 1 : Hybrid avec titre de référence + utilisateur")
print("="*70)

active_users = df.groupby("user_idx").size()
test_user = int(active_users[active_users > 5].index[0])
print(f"\nUser Index: {test_user} ({active_users[test_user]} livres lus)")

ref_book = list(book_idx_to_row.keys())[0]
ref_title_hybrid = idx_to_title.get(ref_book, "")
print(f"Livre de référence : '{ref_title_hybrid}' (book_idx={ref_book})")

hybrid_recs = get_hybrid_recommendations(
    user_idx=test_user,
    title_query=ref_title_hybrid,
    top_n=10,
)
print(f"\nTop 10 Recommendations (Hybrid):")
print(hybrid_recs[["title", "score_cb", "score_cf", "score_mf", "hybrid_score"]].to_string(index=False))

print("\n" + "="*70)
print("Test 2 : Hybrid sans titre (MF + CF fallback uniquement)")
print("="*70)
hybrid_recs_mf = get_hybrid_recommendations(user_idx=test_user, top_n=10)
print(f"\nTop 10 (MF + CF fallback, sans CB):")
print(hybrid_recs_mf[["title", "score_cf", "score_mf", "hybrid_score"]].to_string(index=False))

# ── 5.5  Verification rapide ───────────────────────────────────────────────────────
books_read_set = set(interactions_matrix[test_user].nonzero()[1])
already_read_in_top10 = int(any(b in books_read_set for b in hybrid_recs["book_idx"]))
score_min = float(hybrid_recs["hybrid_score"].min()) if not hybrid_recs.empty else float("nan")
score_max = float(hybrid_recs["hybrid_score"].max()) if not hybrid_recs.empty else float("nan")
print(
    "\nVerification rapide hybrid: "
    f"n_results={len(hybrid_recs)} | "
    f"score_range=[{score_min:.4f}, {score_max:.4f}] | "
    f"livre_deja_lu_present={already_read_in_top10}"
)

# ── 5.6  Visualisation comparée des 3 moteurs ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Comparaison des scores — User {test_user} | Ref: '{ref_title_hybrid[:30]}'",
             fontweight="bold", fontsize=13)

moteurs = [
    ("Content-Based (CB)", "score_cb", "#4C72B0"),
    ("Collaborative (CF)", "score_cf", "#55A868"),
    ("Matrix Factor. (MF)", "score_mf", "#C44E52"),
]

for ax, (label, col, color) in zip(axes, moteurs):
    top5 = hybrid_recs.nlargest(5, col)
    ax.barh(
        top5["title"].str.slice(0, 28).str.cat(["…"] * 5),
        top5[col],
        color=color, edgecolor="white", alpha=0.85
    )
    ax.invert_yaxis()
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Score normalisé [0,1]")
    ax.set_xlim(0, 1.0)

plt.tight_layout()
plt.savefig("hybrid_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure sauvegardée : hybrid_comparison.png")

print(f"\n{'='*70}")
print(" CELLULE 5 TERMINÉE — Hybrid Scorer opérationnel")
print(f"   Poids : CB={W_CB} | CF={W_CF} | MF={W_MF}")
print(f"   Couverture : {n_books} livres | {n_users:,} utilisateurs")
print(f"{'='*70}")

---

## Cellule 10-bis — Évaluation Rigoureuse : Train/Test Split & Métriques Standard

### Objectif
Corriger la faille méthodologique majeure : **tous les modèles étaient évalués sur les données d'entraînement**.

On va maintenant :
1. **Séparer les données** en train (80%) / test (20%) — stratifié par utilisateur
2. **Ré-entraîner tous les modèles** sur le train uniquement
3. **Évaluer sur le test** avec des métriques standard de recommandation

### Métriques utilisées

| Métrique | Pour qui ? | Mesure |
|---|---|---|
| **RMSE / MAE** | MF (SVD) | Précision des notes prédites |
| **Precision@K** | Tous | % de recommandations pertinentes dans le Top-K |
| **Recall@K** | Tous | % des items test retrouvés dans le Top-K |
| **NDCG@K** | Tous | Qualité du ranking (pondéré par position) |
| **Coverage** | Tous | % de livres différents recommandés (diversité) |

### Définition du pertinent
Un livre est **pertinent** si l'utilisateur l'a noté **≥ 4** dans le test set.
C'est le standard utilisé dans les systèmes de recommandation industriels.

---

## Cellule 10-ter — Optimisation des Poids Hybrides (Grid Search)

### Probleme
Les poids hybrides `W_CB=0.30, W_CF=0.40, W_MF=0.30` étaient **arbitraires** — aucune justification empirique.

### Solution : Grid Search
On va tester systematiquement toutes les combinaisons de poids possibles et choisir celles qui maximisent **NDCG@10 sur le test set**.

### Pourquoi NDCG@10 comme metrique d'optimisation ?
- NDCG prend en compte **la position** des items pertinents (un pertinent en position 1 vaut plus qu'en position 10)
- C'est la metrique standard pour evaluer la qualite du ranking dans l'industrie
- Plus robuste que la precision pure car elle penalise les bons resultats places trop bas

### Espace de recherche
On teste des pas de 0.1 : `(W_CF, W_MF)` avec `W_CB = 1 - W_CF - W_MF`

Si `W_CB <= 0` (cas realiste car metadata = 10.6%), on redistribue : `W_CF + W_MF = 1.0`